In [61]:
# Run for ALL weeks

import re
from pathlib import Path
import pandas as pd

WEEK_ROOT = Path("week_folders")
MASTER_HISTORY = Path("historical_matchups.csv")

ROUND_FILE_RE = re.compile(r"(?i)^(?:r|round)\s*(\d+)\.txt$")

# points + optional ID token at end
LINE_RE = re.compile(
    r"""^
    \s*(?P<table>\d*)\s+
    (?P<player>.+?)\s{2,}
    (?P<opponent>.+?)\s{2,}
    (?P<points>\d+\s*-\s*\d+|\d+)
    (?:\s+(?P<idflag>ID))?
    \s*$
    """,
    re.VERBOSE,
)

def split_points(points_raw: str):
    s = str(points_raw).replace(" ", "")
    if "-" in s:
        a, b = s.split("-", 1)
        return int(a), int(b)
    # bye lines often show "6"
    return int(s), None

def outcome_from_points(a: int | None, b: int | None):
    if a is None or b is None:
        return None
    if a > b:
        return 1.0
    if a < b:
        return 0.0
    return 0.5

def find_week_folders(root: Path) -> list[Path]:
    return sorted([p for p in root.iterdir() if p.is_dir()])

def find_round_files(week_folder: Path) -> list[tuple[int, Path]]:
    found = []
    for p in week_folder.iterdir():
        if p.is_file() and p.suffix.lower() == ".txt":
            m = ROUND_FILE_RE.match(p.name)
            if m:
                found.append((int(m.group(1)), p))
    return sorted(found, key=lambda x: x[0])

def parse_round_file(path: Path, round_num: int):
    """
    Parse raw lines into rows. We DO NOT decide Win/Loss/Draw here unless it's per-match mode.
    We store totals (left/right) so we can compute deltas later if cumulative mode.
    """
    rows = []
    seq = 0
    for raw in path.read_text(encoding="utf-8", errors="replace").splitlines():
        line = raw.rstrip()
        if not line.strip():
            continue
        if "Table" in line or set(line.strip()) <= {"-"}:
            continue
        if line.strip().lower().startswith("eventlink"):
            continue
        if "copyright" in line.lower():
            continue
        if line.strip().lower().startswith("round"):
            continue

        m = LINE_RE.match(line)
        if not m:
            continue

        seq += 1
        player = m.group("player").strip()
        opponent = m.group("opponent").strip()
        points_raw = m.group("points").strip()
        is_id = (m.group("idflag") is not None)
        is_bye = (opponent == "***Bye***")

        left, right = split_points(points_raw)

        rows.append({
            "Round": round_num,
            "Seq": seq,  # preserves file order within a round
            "Player": player,
            "Opponent": opponent,
            "PointsRaw": points_raw.replace(" ", ""),
            "IsBye": bool(is_bye),
            "IsID": bool(is_id),
            "PlayerTotal": left,
            "OppTotal": right,  # None for byes or malformed
            # ScoreSelf computed later
        })

    return rows

def compute_scores(df_rows: pd.DataFrame) -> pd.DataFrame:
    df = df_rows.copy()
    df["Round"] = df["Round"].astype(int)
    df["Seq"] = df["Seq"].astype(int)

    non_bye = df[~df["IsBye"]].copy()
    max_seen = 0
    if not non_bye.empty:
        max_seen = max(
            pd.to_numeric(non_bye["PlayerTotal"], errors="coerce").max(skipna=True),
            pd.to_numeric(non_bye["OppTotal"], errors="coerce").max(skipna=True),
        )
        max_seen = int(max_seen) if pd.notna(max_seen) else 0

    cumulative_mode = (max_seen > 3)
    df["ScoreSelf"] = None

    # -----------------------------
    # Per-match mode (3-0, 1-1 etc)
    # -----------------------------
    if not cumulative_mode:
        for idx, row in df.iterrows():
            if row["IsBye"]:
                df.at[idx, "ScoreSelf"] = None
                continue
            if row["IsID"]:
                df.at[idx, "ScoreSelf"] = 0.5
                continue
            df.at[idx, "ScoreSelf"] = outcome_from_points(row["PlayerTotal"], row["OppTotal"])
        return df

    # -----------------------------
    # CUMULATIVE MODE (13-10 etc)
    # FIX: compute per-round using prev snapshot, not per-row updates
    # -----------------------------
    prev = {}  # player -> previous cumulative total

    # process round by round
    for rnd in sorted(df["Round"].unique()):
        r = df[df["Round"] == rnd].copy().sort_values("Seq")

        # 1) snapshot prev at start of round
        prev_snapshot = dict(prev)

        # 2) apply byes: printed number is meaningless; bye = +3
        # (byes have no matchup rows)
        bye_rows = r[r["IsBye"]]
        for _, row in bye_rows.iterrows():
            p = str(row["Player"]).strip()
            prev[p] = int(prev.get(p, 0)) + 3

        # 3) build new totals for everyone who played this round
        new_total = {}  # name -> cumulative total after this round
        play_rows = r[~r["IsBye"]]

        for _, row in play_rows.iterrows():
            p = str(row["Player"]).strip()
            o = str(row["Opponent"]).strip()

            new_total[p] = int(row["PlayerTotal"])
            if pd.notna(row["OppTotal"]):
                new_total[o] = int(row["OppTotal"])

        # 4) compute ScoreSelf for each row using prev_snapshot (NOT updated mid-round)
        for idx, row in play_rows.iterrows():
            p = str(row["Player"]).strip()
            o = str(row["Opponent"]).strip()

            if row["IsID"]:
                df.at[idx, "ScoreSelf"] = 0.5
                continue

            p_new = new_total.get(p, None)
            o_new = new_total.get(o, None)

            if p_new is None or o_new is None:
                df.at[idx, "ScoreSelf"] = None
                continue

            p_prev = int(prev_snapshot.get(p, 0))
            o_prev = int(prev_snapshot.get(o, 0))

            p_gain = p_new - p_prev
            o_gain = o_new - o_prev

            if p_gain == 3 and o_gain == 0:
                df.at[idx, "ScoreSelf"] = 1.0
            elif p_gain == 0 and o_gain == 3:
                df.at[idx, "ScoreSelf"] = 0.0
            elif p_gain == 1 and o_gain == 1:
                df.at[idx, "ScoreSelf"] = 0.5
            else:
                df.at[idx, "ScoreSelf"] = None  # ambiguous, do not force draw

        # 5) after scoring, update prev to the new totals for next rounds
        for name, tot in new_total.items():
            prev[name] = int(tot)

    return df

def collapse_to_matchups(df_rows: pd.DataFrame, week_id: str) -> pd.DataFrame:
    """
    Takes per-row entries (possibly doubled) and returns one row per match.
    - IDs are kept (Status="ID", IsID=True, ScoreA=0.5)
    - Byes are dropped
    - Works whether rows are doubled or not
    """
    df = df_rows.copy()
    df = df[~df["IsBye"]].copy()

    df["Player"] = df["Player"].astype(str).str.strip()
    df["Opponent"] = df["Opponent"].astype(str).str.strip()
    df["Round"] = df["Round"].astype(int)

    # Canonical match key per round
    lo = df[["Player", "Opponent"]].min(axis=1)
    hi = df[["Player", "Opponent"]].max(axis=1)
    df["P_lo"] = lo
    df["P_hi"] = hi

    out = []
    for (rnd, p_lo, p_hi), g in df.groupby(["Round", "P_lo", "P_hi"], sort=True):
        row_lo = g[g["Player"] == p_lo]
        row_hi = g[g["Player"] == p_hi]

        # Representative score for PlayerA (=p_lo)
        if len(row_lo) >= 1:
            rA = row_lo.iloc[0]
            score_a = rA["ScoreSelf"]
        else:
            rB = row_hi.iloc[0]
            score_b = rB["ScoreSelf"]
            score_a = None if score_b is None else (1.0 - float(score_b))

        is_id = bool(g["IsID"].any()) or (str(g.get("Status", "")).strip() == "ID")

        if is_id:
            score_a = 0.5
            status = "ID"
        else:
            status = "OK" if score_a in (0.0, 0.5, 1.0) else "NeedsReview"

            # if doubled and both directions present, sanity check
            if len(row_lo) == 1 and len(row_hi) == 1:
                a_score = row_lo.iloc[0]["ScoreSelf"]
                b_score = row_hi.iloc[0]["ScoreSelf"]
                if (a_score in (0.0,0.5,1.0)) and (b_score in (0.0,0.5,1.0)):
                    if abs((float(a_score) + float(b_score)) - 1.0) > 1e-9 and not (a_score == b_score == 0.5):
                        status = "CheckOutcomeMismatch"

        out.append({
            "Week": week_id,
            "Round": int(rnd),
            "PlayerA": p_lo,
            "PlayerB": p_hi,
            "ScoreA": float(score_a) if score_a is not None else None,
            "Status": status,
            "IsID": bool(is_id),
        })

    return pd.DataFrame(out).sort_values(["Week", "Round", "PlayerA", "PlayerB"]).reset_index(drop=True)

def extract_byes(df_rows: pd.DataFrame, week_id: str) -> pd.DataFrame:
    """
    Turn bye rows into matchup-like rows so player pages can display them.
    These should NOT affect Elo.
    """
    dfb = df_rows[df_rows["IsBye"]].copy()
    if dfb.empty:
        return pd.DataFrame(columns=["Week","Round","PlayerA","PlayerB","ScoreA","Status","IsID","IsBye"])

    out = pd.DataFrame({
        "Week": week_id,
        "Round": dfb["Round"].astype(int),
        "PlayerA": dfb["Player"].astype(str).str.strip(),
        "PlayerB": "***Bye***",
        "ScoreA": None,
        "Status": "BYE",
        "IsID": False,
        "IsBye": True,
    })
    return out.sort_values(["Week","Round","PlayerA"]).reset_index(drop=True)

def build_master_history_from_week_root(week_root: Path) -> pd.DataFrame:
    all_matchups = []

    for week_folder in find_week_folders(week_root):
        week_id = week_folder.name
        round_files = find_round_files(week_folder)
        if not round_files:
            continue

        rows = []
        for rnd, path in round_files:
            rows.extend(parse_round_file(path, rnd))

        df_rows = pd.DataFrame(rows)
        if df_rows.empty:
            continue

        # NEW: compute ScoreSelf correctly (auto-detect cumulative vs per-match)
        df_rows = compute_scores(df_rows)

        df_matchups = collapse_to_matchups(df_rows, week_id=week_id)
        df_byes = extract_byes(df_rows, week_id=week_id)

        all_matchups.append(pd.concat([df_matchups, df_byes], ignore_index=True))

    if not all_matchups:
        raise ValueError(f"No matchups found under {week_root}")

    df_master = pd.concat(all_matchups, ignore_index=True)
    df_master = df_master.sort_values(["Week", "Round", "PlayerA", "PlayerB"]).reset_index(drop=True)
    df_master.to_csv(MASTER_HISTORY, index=False)

    print(f"✅ wrote {MASTER_HISTORY} with {len(df_master)} matches")
    print(df_master["Status"].value_counts(dropna=False).to_string())

    # Optional: show any NeedsReview matches (these should NOT be treated as draws)
    bad = df_master[df_master["Status"] == "NeedsReview"]
    if not bad.empty:
        print("\n[WARN] NeedsReview sample:")
        print(bad.head(20).to_string(index=False))

    return df_master

if __name__ == "__main__":
    build_master_history_from_week_root(WEEK_ROOT)

C:\Users\WILKIAX2\AppData\Local\Temp\ipykernel_22944\2442048829.py:308: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_matchups.append(pd.concat([df_matchups, df_byes], ignore_index=True))
C:\Users\WILKIAX2\AppData\Local\Temp\ipykernel_22944\2442048829.py:308: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_matchups.append(pd.concat([df_matchups, df_byes], ignore_index=True))
C:\Users\WILKIAX2\AppData\Local\Temp\ipykernel_22944\2442048829.py:308: FutureWarning: The behavior of DataFrame con

✅ wrote historical_matchups.csv with 675 matches
Status
OK     644
BYE     19
ID      12


C:\Users\WILKIAX2\AppData\Local\Temp\ipykernel_22944\2442048829.py:308: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_matchups.append(pd.concat([df_matchups, df_byes], ignore_index=True))
C:\Users\WILKIAX2\AppData\Local\Temp\ipykernel_22944\2442048829.py:308: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_matchups.append(pd.concat([df_matchups, df_byes], ignore_index=True))


In [62]:
# print leaderboard
from pathlib import Path
import pandas as pd

# =========================
# 1) Load player_list.csv -> alias map + opt-in set
# =========================
def load_player_list(player_list_csv: str | Path) -> tuple[dict[str, str], set[str]]:
    """
    player_list.csv format:
      canonical_name, alias1, alias2, alias3, ...
    Returns:
      alias_to_canonical: dict mapping any alias/canonical -> canonical
      opt_in_canonicals: set of canonical names (column 1 values)
    """
    df = pd.read_csv(player_list_csv, header=None, dtype=str).fillna("")
    alias_to_canonical: dict[str, str] = {}
    opt_in: set[str] = set()

    for _, row in df.iterrows():
        canonical = str(row.iloc[0]).strip()
        if not canonical:
            continue
        opt_in.add(canonical)

        # Map canonical to itself
        alias_to_canonical[canonical] = canonical

        # Map aliases to canonical
        for cell in row.iloc[1:]:
            alias = str(cell).strip()
            if alias:
                alias_to_canonical[alias] = canonical

    return alias_to_canonical, opt_in


def canon_name(name: str, alias_to_canonical: dict[str, str]) -> str:
    n = str(name).strip()
    return alias_to_canonical.get(n, n)


# =========================
# 2) Elo helpers
# =========================
def expected_score(r_a: float, r_b: float) -> float:
    return 1.0 / (1.0 + 10 ** ((r_b - r_a) / 400.0))

def elo_update(r_a: float, r_b: float, s_a: float, k: float) -> tuple[float, float]:
    e_a = expected_score(r_a, r_b)
    e_b = 1.0 - e_a
    s_b = 1.0 - s_a
    return (r_a + k * (s_a - e_a), r_b + k * (s_b - e_b))


# =========================
# 3) Elo from historical record with:
#    - alias collapsing
#    - opt-in tracking only
#    - non-opt-in opponents always treated as 1500 and NOT updated
# =========================
def run_opt_in_elo_from_master(
    master_history_csv: str | Path,
    player_list_csv: str | Path,
    initial_elo: float = 1500.0,
    k: float = 32.0,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Returns:
      leaderboard (final standings for opt-in players),
      match_history (only matches where at least one opt-in played),
      ladder_by_week (opt-in players only, snapshot each week; carries forward if inactive)
    """
    alias_to_canonical, opt_in = load_player_list(player_list_csv)

    df = pd.read_csv(master_history_csv)
    required = {"Week", "Round", "PlayerA", "PlayerB", "ScoreA", "Status"}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"master history missing columns: {sorted(missing)}")

    # Only use OK matches
    df = df[df["Status"] == "OK"].copy()
    if df.empty:
        raise ValueError("No OK match rows found in master history.")

    # Canonicalize names
    df["Week"] = df["Week"].astype(str)
    df["Round"] = df["Round"].astype(int)
    df["PlayerA"] = df["PlayerA"].astype(str).map(lambda x: canon_name(x, alias_to_canonical))
    df["PlayerB"] = df["PlayerB"].astype(str).map(lambda x: canon_name(x, alias_to_canonical))
    df["ScoreA"] = df["ScoreA"].astype(float)

    # Deterministic order
    df = df.sort_values(["Week", "Round", "PlayerA", "PlayerB"]).reset_index(drop=True)

    # Ratings only for opt-in canonicals
    ratings: dict[str, float] = {p: float(initial_elo) for p in opt_in}
    stats: dict[str, dict[str, int]] = {p: {"Matches": 0, "Wins": 0, "Losses": 0, "Draws": 0} for p in opt_in}

    # Weeks in data
    weeks = list(dict.fromkeys(df["Week"].tolist()))

    match_history_rows = []
    ladder_rows = []
    match_index = 0

    def get_rating_for_player(p: str) -> float:
        # opt-in: current rating; non-opt-in: always 1500
        return ratings[p] if p in opt_in else float(initial_elo)

    for wk in weeks:
        df_w = df[df["Week"] == wk]

        # Process matches in this week
        for _, m in df_w.iterrows():
            a = m["PlayerA"]
            b = m["PlayerB"]
            s_a = float(m["ScoreA"])

            a_in = (a in opt_in)
            b_in = (b in opt_in)

            # Skip match entirely if neither player is opt-in
            if not (a_in or b_in):
                continue

            pre_a = get_rating_for_player(a)
            pre_b = get_rating_for_player(b)

            post_a, post_b = elo_update(pre_a, pre_b, s_a, k)

            # Update ONLY opt-in ratings
            if a_in:
                ratings[a] = post_a
            if b_in:
                ratings[b] = post_b

            # Update ONLY opt-in stats
            if a_in:
                stats[a]["Matches"] += 1
                if s_a == 1.0:
                    stats[a]["Wins"] += 1
                elif s_a == 0.0:
                    stats[a]["Losses"] += 1
                else:
                    stats[a]["Draws"] += 1

            if b_in:
                stats[b]["Matches"] += 1
                s_b = 1.0 - s_a
                if s_b == 1.0:
                    stats[b]["Wins"] += 1
                elif s_b == 0.0:
                    stats[b]["Losses"] += 1
                else:
                    stats[b]["Draws"] += 1

            match_index += 1
            match_history_rows.append({
                "MatchIndex": match_index,
                "Week": wk,
                "Round": int(m["Round"]),
                "PlayerA": a,
                "PlayerB": b,
                "ScoreA": s_a,
                "PreA": pre_a,
                "PreB": pre_b,
                "PostA": post_a if a_in else None,   # None because we don't track non-opt-in updates
                "PostB": post_b if b_in else None,
                "A_OptIn": a_in,
                "B_OptIn": b_in,
                "K": k,
                "NonOptInOppRatingUsed": initial_elo,  # explicit
            })

        # Snapshot: EVERY opt-in player each week (even if they didn't play)
        for p in sorted(opt_in):
            ladder_rows.append({
                "Week": wk,
                "Player": p,
                "Elo": ratings.get(p, float(initial_elo)),
                "Matches": stats[p]["Matches"],
                "Wins": stats[p]["Wins"],
                "Losses": stats[p]["Losses"],
                "Draws": stats[p]["Draws"],
            })

    match_history = pd.DataFrame(match_history_rows)
    ladder_by_week = pd.DataFrame(ladder_rows).sort_values(["Player", "Week"]).reset_index(drop=True)

    ladder_by_week["DeltaElo"] = ladder_by_week.groupby("Player")["Elo"].diff().fillna(0.0)
    ladder_by_week["Elo"] = ladder_by_week["Elo"].round(2)
    ladder_by_week["DeltaElo"] = ladder_by_week["DeltaElo"].round(2)

    # Final leaderboard = last week snapshot
    last_week = weeks[-1]
    leaderboard = (
        ladder_by_week[ladder_by_week["Week"] == last_week]
        .sort_values(["Elo", "Wins"], ascending=[False, False])
        .reset_index(drop=True)
    )

    return leaderboard, match_history, ladder_by_week


def print_leaders(leaderboard: pd.DataFrame, top_n: int = 15):
    cols = ["Player", "Elo", "Matches", "Wins", "Losses", "Draws"]
    print(leaderboard[cols].head(top_n).to_string(index=False))


# ==========
# RUN THIS ANYTIME (RECOMPUTES FROM SCRATCH)
# ==========
MASTER_HISTORY = "historical_matchups.csv"
PLAYER_LIST = "player_list.csv"

leaderboard, match_history, ladder_by_week = run_opt_in_elo_from_master(
    master_history_csv=MASTER_HISTORY,
    player_list_csv=PLAYER_LIST,
    initial_elo=1500,
    k=32,
)

print_leaders(leaderboard, top_n=50)

leaderboard.to_csv("elo_leaderboard.csv", index=False)
match_history.to_csv("elo_match_history.csv", index=False)
ladder_by_week.to_csv("elo_ladder_by_week.csv", index=False)

              Player     Elo  Matches  Wins  Losses  Draws
   Anthony Wilkinson 1654.03       40    30       9      1
      David Van Hise 1648.55       34    24      10      0
         Nathan Choi 1637.09       36    24      10      2
         Logan Hesch 1636.33       30    20      10      0
          Amy Altman 1606.87       28    19       9      0
      Bradley Skubic 1602.88       13    10       2      1
     Kenton Najdzien 1596.36       19    13       6      0
     Becca Blackwood 1594.97       28    18      10      0
          Dani Masom 1571.54       39    20      18      1
          Milo Brown 1559.73       23    15       7      1
      Kevin Thanakit 1556.92        5     4       1      0
       Bailey Sarkis 1550.42        5     4       1      0
       Stan Kornacki 1535.85       34    21      12      1
             Ran Guo 1533.34        6     4       2      0
   Isharah Considine 1531.50       18    10       8      0
          Joel Brook 1531.30        6     4       2     

In [63]:
# Update Elo Leaderboard
import pandas as pd
from pathlib import Path
import json

# =========================
# CONFIG (EDIT THESE)
# =========================
MASTER_HISTORY_CSV = Path("historical_matchups.csv")     # your master matchups
PLAYER_LIST_CSV    = Path("player_list.csv")            # canonical,alias1,alias2,...
OUT_LEADERBOARD_JSON = Path("docs/data/leaderboard.json")

INITIAL_ELO = 1500.0
K_FACTOR = 38.0


# =========================
# Player list -> alias map
# =========================
def _norm(s: str) -> str:
    return str(s).strip().casefold()

def is_win(x: float) -> bool:
    return abs(x - 1.0) < 1e-9

def is_loss(x: float) -> bool:
    return abs(x - 0.0) < 1e-9

def is_draw(x: float) -> bool:
    return abs(x - 0.5) < 1e-9

def load_player_list(player_list_csv: Path) -> tuple[dict[str, str], set[str]]:
    df = pd.read_csv(player_list_csv, header=None, dtype=str).fillna("")
    alias_to_canonical: dict[str, str] = {}
    opt_in: set[str] = set()

    for _, row in df.iterrows():
        canonical = str(row.iloc[0]).strip()
        if not canonical:
            continue

        opt_in.add(canonical)

        # Map canonical + aliases using normalized keys
        alias_to_canonical[_norm(canonical)] = canonical
        for cell in row.iloc[1:]:
            alias = str(cell).strip()
            if alias:
                alias_to_canonical[_norm(alias)] = canonical

    return alias_to_canonical, opt_in

def canon_name(name: str, alias_to_canonical: dict[str, str]) -> str:
    raw = str(name).strip()
    return alias_to_canonical.get(_norm(raw), raw)


# =========================
# Elo helpers
# =========================
ELO_SCALE = 1135.77  # 200 pts -> ~60% win expectancy

def expected_score(r_a: float, r_b: float, scale: float = ELO_SCALE) -> float:
    # (1 + 10^((R_B - R_A)/scale))^-1
    return 1.0 / (1.0 + 10 ** ((r_b - r_a) / scale))

def elo_update(r_a: float, r_b: float, s_a: float, k: float, scale: float = ELO_SCALE):
    e_a = expected_score(r_a, r_b, scale=scale)
    e_b = 1.0 - e_a
    s_b = 1.0 - s_a
    return (r_a + k * (s_a - e_a), r_b + k * (s_b - e_b))

# =========================
# Main: compute opt-in Elo + export leaderboard.json
# =========================
def export_leaderboard_json(
    master_history_csv: Path,
    player_list_csv: Path,
    out_json: Path,
    initial_elo: float = 1500.0,
    k: float = 38.0,
):
    alias_to_canonical, opt_in = load_player_list(player_list_csv)

    df = pd.read_csv(master_history_csv)

    required = {"Week", "Round", "PlayerA", "PlayerB", "ScoreA", "Status"}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"master_history missing columns: {sorted(missing)}")

    # Only OK matches (byes already excluded upstream)
    df = df[df["Status"].isin(["OK", "ID"])].copy()
    if df.empty:
        raise ValueError("No OK matches found in master history. Check Status values and matchup generation.")

    # Canonicalize names
    df["Week"] = df["Week"].astype(str)
    df["Round"] = df["Round"].astype(int)
    df["PlayerA"] = df["PlayerA"].astype(str).map(lambda x: canon_name(x, alias_to_canonical))
    df["PlayerB"] = df["PlayerB"].astype(str).map(lambda x: canon_name(x, alias_to_canonical))
    df["ScoreA"] = df["ScoreA"].astype(float)

    # Deterministic order
    df = df.sort_values(["Week", "Round", "PlayerA", "PlayerB"]).reset_index(drop=True)

    # Ratings/stats only for opt-in players
    ratings = {p: float(initial_elo) for p in opt_in}
    stats = {p: {"matches": 0, "wins": 0, "losses": 0, "draws": 0} for p in opt_in}

    def rating_for(p: str) -> float:
        # Opt-in: current rating; non-opt-in: fixed neutral
        return ratings[p] if p in opt_in else float(initial_elo)

    # Iterate matches
    for _, m in df.iterrows():
        a = m["PlayerA"]
        b = m["PlayerB"]
        s_a = float(m["ScoreA"])

        a_in = a in opt_in
        b_in = b in opt_in

        # Ignore matches where neither is opt-in
        if not (a_in or b_in):
            continue

        pre_a = rating_for(a)
        pre_b = rating_for(b)

        # NEW: detect Intentional Draw
        is_id = bool(m.get("IsID", False)) or (str(m.get("Status", "")).strip() == "ID")

        if is_id:
            # ID: no Elo change
            post_a, post_b = pre_a, pre_b
        else:
            post_a, post_b = elo_update(pre_a, pre_b, s_a, k)

        # Update ONLY opt-in ratings (skip update for ID)
        if not is_id:
            if a_in:
                ratings[a] = post_a
            if b_in:
                ratings[b] = post_b

        # Update ONLY opt-in stats
        # Update ONLY opt-in stats
        if a_in:
            stats[a]["matches"] += 1
            if is_id:
                stats[a]["draws"] += 1
            else:
                if is_win(s_a):
                    stats[a]["wins"] += 1
                elif is_loss(s_a):
                    stats[a]["losses"] += 1
                elif is_draw(s_a):
                    stats[a]["draws"] += 1
                else:
                    # bad score; don't count as draw
                    pass

        if b_in:
            stats[b]["matches"] += 1
            if is_id:
                stats[b]["draws"] += 1
            else:
                s_b = 1.0 - s_a
                if is_win(s_b):
                    stats[b]["wins"] += 1
                elif is_loss(s_b):
                    stats[b]["losses"] += 1
                elif is_draw(s_b):
                    stats[b]["draws"] += 1
                else:
                    pass

    # Build leaderboard rows in the exact format your HTML expects
    rows = []
    for p in sorted(opt_in):
        rows.append({
            "player": p,
            "elo": round(float(ratings.get(p, initial_elo)), 2),
            "matches": int(stats[p]["matches"]),
            "wins": int(stats[p]["wins"]),
            "losses": int(stats[p]["losses"]),
            "draws": int(stats[p]["draws"]),
        })

    # Sort by Elo desc, wins desc
    rows.sort(key=lambda r: (r["elo"], r["wins"]), reverse=True)

    out_json.parent.mkdir(parents=True, exist_ok=True)
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)

    print(f"✅ Wrote {out_json} ({len(rows)} players)")
    print("Top 10:")
    for r in rows[:10]:
        print(f"  {r['player']:<24} Elo={r['elo']:<7} W-L-T={r['wins']}-{r['losses']}-{r['draws']}  Games={r['matches']}")


# =========================
# RUN
# =========================
if __name__ == "__main__":
    export_leaderboard_json(
        master_history_csv=MASTER_HISTORY_CSV,
        player_list_csv=PLAYER_LIST_CSV,
        out_json=OUT_LEADERBOARD_JSON,
        initial_elo=INITIAL_ELO,
        k=K_FACTOR,
    )

✅ Wrote docs\data\leaderboard.json (45 players)
Top 10:
  Anthony Wilkinson        Elo=1779.76 W-L-T=30-9-2  Games=41
  David Van Hise           Elo=1721.31 W-L-T=24-10-0  Games=34
  Nathan Choi              Elo=1708.89 W-L-T=24-10-2  Games=36
  Logan Hesch              Elo=1675.71 W-L-T=20-10-0  Games=30
  Amy Altman               Elo=1659.43 W-L-T=19-9-3  Games=31
  Bradley Skubic           Elo=1638.11 W-L-T=10-2-4  Games=16
  Becca Blackwood          Elo=1634.35 W-L-T=18-10-2  Games=30
  Kenton Najdzien          Elo=1626.75 W-L-T=13-6-1  Games=20
  Milo Brown               Elo=1609.59 W-L-T=15-7-4  Games=26
  Stan Kornacki            Elo=1595.68 W-L-T=21-12-3  Games=36


In [64]:
# Update Player Match Histories
# =========================
# CONFIG (EDIT)
# =========================
MASTER_HISTORY_CSV = Path("historical_matchups.csv")
PLAYER_LIST_CSV = Path("player_list.csv")

SITE_DATA_DIR = Path("docs/data")
LEADERBOARD_JSON = SITE_DATA_DIR / "leaderboard.json"
PLAYERS_DIR = SITE_DATA_DIR / "players"
PLAYERS_INDEX_JSON = SITE_DATA_DIR / "players_index.json"

INITIAL_ELO = 1500.0
K_FACTOR = 38.0

# =========================
# Helpers
# =========================
def slugify(name: str) -> str:
    s = str(name).lower().strip()
    s = re.sub(r"[\'\"]", "", s)
    s = re.sub(r"[^a-z0-9]+", "-", s)
    s = re.sub(r"^-+|-+$", "", s)
    return s

def _norm(s: str) -> str:
    return str(s).strip().casefold()

def canon_name(name: str, alias_to_canonical: dict[str, str]) -> str:
    raw = str(name).strip()
    return alias_to_canonical.get(_norm(raw), raw)

def expected_score(r_a: float, r_b: float) -> float:
    return 1.0 / (1.0 + 10 ** ((r_b - r_a) / 400.0))

def elo_update(r_a: float, r_b: float, s_a: float, k: float):
    e_a = expected_score(r_a, r_b)
    e_b = 1.0 - e_a
    s_b = 1.0 - s_a
    return (r_a + k * (s_a - e_a), r_b + k * (s_b - e_b))

# =========================
# Build site data
# =========================
def export_site_data():
    alias_to_canonical, opt_in = load_player_list(PLAYER_LIST_CSV)
    opt_in_norm = {_norm(p) for p in opt_in}

    df = pd.read_csv(MASTER_HISTORY_CSV)
    required = {"Week", "Round", "PlayerA", "PlayerB", "ScoreA", "Status"}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"historical_matchups.csv missing columns: {sorted(missing)}")

    df = df[df["Status"].isin(["OK", "ID", "BYE"])].copy()
    if df.empty:
        raise ValueError("No OK matches found in historical_matchups.csv")

    df["Week"] = df["Week"].astype(str)
    df["Round"] = df["Round"].astype(int)
    df["PlayerA"] = df["PlayerA"].astype(str).map(lambda x: canon_name(x, alias_to_canonical))
    df["PlayerB"] = df["PlayerB"].astype(str).map(lambda x: canon_name(x, alias_to_canonical))
    df["ScoreA"] = df["ScoreA"].astype(float)

    # deterministic processing order
    df = df.sort_values(["Week", "Round", "PlayerA", "PlayerB"]).reset_index(drop=True)

    # ratings/stats only for opt-in players
    ratings = {p: float(INITIAL_ELO) for p in opt_in}
    stats = {p: {"matches": 0, "wins": 0, "losses": 0, "draws": 0} for p in opt_in}
    player_matches = {p: [] for p in opt_in}

    def rating_for_expectation(p: str) -> float:
        # opt-in: true rating, otherwise ALWAYS treated as 1500
        return ratings[p] if p in opt_in else float(INITIAL_ELO)

    for _, m in df.iterrows():
        a = m["PlayerA"]
        b = m["PlayerB"]
        s_a = float(m["ScoreA"])

        a = canon_name(a, alias_to_canonical)
        b = canon_name(b, alias_to_canonical)

        a_in = a in opt_in
        b_in = b in opt_in

        # ignore matches where neither is opt-in
        if not (a_in or b_in):
            continue

        # Pre-match ratings used for expectation (and what we show in match history)
        pre_a = rating_for_expectation(a)
        pre_b = rating_for_expectation(b)
        is_id = bool(m.get("IsID", False)) or (str(m.get("Status", "")).strip() == "ID")
        is_bye = (str(m.get("Status", "")).strip().upper() == "BYE") or (str(b).strip() == "***Bye***")

        # --- If BYE: add a display row, but DO NOT affect Elo or counts ---
        if is_bye:
            if a_in:
                # BYE row: visible only; no stats increment; no rating change
                player_matches[a].append({
                    "week": week,
                    "round": int(rnd),
                    "opponent": "Bye",
                    "result": "BYE",
                    "is_bye": True,
                    "elo_before": round(float(pre_a), 4),
                    "elo_after": round(float(pre_a), 4),
                    "opp_elo": None,
                })
            continue

        if is_bye or is_id:
            post_a, post_b = pre_a, pre_b  # no Elo change
        else:
            post_a, post_b = elo_update(pre_a, pre_b, s_a, K_FACTOR)

        # Only update ratings if NOT ID and NOT BYE
        if (not is_id) and (not is_bye):
            if a_in:
                ratings[a] = post_a
            if b_in:
                ratings[b] = post_b

        # Only update ratings if NOT ID
        if not is_id:
            if a_in:
                ratings[a] = post_a
            if b_in:
                ratings[b] = post_b

        # Helper: append match rows (including opponent elo at the time)
        def add_row(player: str, opponent: str, score: float,
                    player_pre: float, player_post: float,
                    opp_elo: float,
                    week: str, rnd: int, is_id: bool):
            if player not in opt_in:
                return
            if is_bye:
                res = "BYE"
            elif is_id:
                res = "ID"
            else:
                res = "W" if score == 1.0 else "L" if score == 0.0 else "D"
            player_matches[player].append({
                "week": week,
                "round": int(rnd),
                "opponent": opponent,
                "result": res,
                "score": score,
                "elo_before": round(float(player_pre), 4),
                "elo_after": round(float(player_post), 4),
                "opp_elo": None if is_bye else round(float(opp_elo), 4),
                "is_bye": bool(is_bye)
            })

        week = m["Week"]
        rnd = int(m["Round"])


        # Player A perspective
        if a_in:
            stats[a]["matches"] += 1
            if is_id:
                stats[a]["draws"] += 1
            else:
                # normal win/loss/draw based on score
                if s_a == 1.0:
                    stats[a]["wins"] += 1
                elif s_a == 0.0:
                    stats[a]["losses"] += 1
                else:
                    stats[a]["draws"] += 1

            add_row(
                player=a, opponent=b, score=s_a,
                player_pre=pre_a, player_post=post_a,
                opp_elo=pre_b,
                week=week, rnd=rnd,
                is_id=is_id
            )

        # Player B perspective (ScoreB = 1 - ScoreA)
        if b_in:
            s_b = 1.0 - s_a
            stats[b]["matches"] += 1
            if is_id:
                stats[b]["draws"] += 1
            else: 
                if s_b == 1.0:
                    stats[b]["wins"] += 1
                elif s_b == 0.0:
                    stats[b]["losses"] += 1
                else:
                    stats[b]["draws"] += 1

            add_row(
                player=b, opponent=a, score=s_b,
                player_pre=pre_b, player_post=post_b,
                opp_elo=pre_a, 
                week=week, rnd=rnd,
                is_id=is_id
            )

    # Ensure output dirs
    SITE_DATA_DIR.mkdir(parents=True, exist_ok=True)
    PLAYERS_DIR.mkdir(parents=True, exist_ok=True)

    # Leaderboard JSON (NOTE draws)
    leaderboard_rows = []
    for p in sorted(opt_in):
        leaderboard_rows.append({
            "player": p,
            "elo": round(float(ratings.get(p, INITIAL_ELO)), 2),
            "matches": int(stats[p]["matches"]),
            "wins": int(stats[p]["wins"]),
            "losses": int(stats[p]["losses"]),
            "draws": int(stats[p]["draws"]),
        })

    leaderboard_rows.sort(key=lambda r: (r["elo"], r["wins"]), reverse=True)
    LEADERBOARD_JSON.write_text(json.dumps(leaderboard_rows, ensure_ascii=False, indent=2), encoding="utf-8")
    
    OUT_CSV = Path("elo_leaderboard.csv")

    # leaderboard_rows already exists here
    pd.DataFrame(leaderboard_rows).to_csv(OUT_CSV, index=False)
    print(f"✅ wrote {OUT_CSV}")

    # Per-player JSON + index
    players_index = []
    for p in sorted(opt_in):
        slug = slugify(p)
        pdata = {
            "player": p,
            "slug": slug,
            "elo": round(float(ratings.get(p, INITIAL_ELO)), 2),
            "matches": int(stats[p]["matches"]),
            "wins": int(stats[p]["wins"]),
            "losses": int(stats[p]["losses"]),
            "draws": int(stats[p]["draws"]),
            "matches": player_matches.get(p, []),
        }
        (PLAYERS_DIR / f"{slug}.json").write_text(json.dumps(pdata, ensure_ascii=False, indent=2), encoding="utf-8")
        players_index.append({"player": p, "slug": slug})

    PLAYERS_INDEX_JSON.write_text(json.dumps(players_index, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"✅ wrote {LEADERBOARD_JSON}")
    print(f"✅ wrote {PLAYERS_INDEX_JSON}")
    print(f"✅ wrote {len(players_index)} player JSON files into {PLAYERS_DIR}/")

if __name__ == "__main__":
    export_site_data()

✅ wrote elo_leaderboard.csv
✅ wrote docs\data\leaderboard.json
✅ wrote docs\data\players_index.json
✅ wrote 45 player JSON files into docs\data\players/
